In [ ]:
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager

# --- SENSITIVE DATA REDACTED ---
SERVER_IP = "123.456.78.90"  # Replace with the real IP only for local use
PORT = "18881"
TARGET_URL = f"http://{SERVER_IP}:{PORT}/viewer?url=..."

def download_medical_images_anonymized(url):
    # Setup download directory
    download_dir = os.path.join(os.getcwd(), "MEDICAL_DOWNLOADS")
    if not os.path.exists(download_dir): os.makedirs(download_dir)

    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")

    # Bypass security for Insecure Content (HTTP Blobs)
    chrome_options.add_argument("--allow-running-insecure-content")
    chrome_options.add_argument("--ignore-certificate-errors")
    chrome_options.add_argument(f"--unsafely-treat-insecure-origin-as-secure=http://{SERVER_IP}:{PORT}")

    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "profile.default_content_setting_values.automatic_downloads": 1,
        "safebrowsing.enabled": False
    }
    chrome_options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    wait = WebDriverWait(driver, 30)

    try:
        print(f"🚀 Loading PACS Viewer from {SERVER_IP}...")
        driver.get(url)
        time.sleep(25) # Wait for Cornerstone.js engine to initialize

        # Switch to the main iframe where the viewer lives
        iframes = driver.find_elements(By.TAG_NAME, "iframe")
        if iframes: driver.switch_to.frame(iframes[0])

        # Detect image series/folders
        series = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.thumbnail, [data-cy*="thumbnail"]')))

        for s_idx in range(len(series)):
            series = driver.find_elements(By.CSS_SELECTOR, '.thumbnail, [data-cy*="thumbnail"]')
            print(f"📂 Processing Series {s_idx+1}")
            driver.execute_script("arguments[0].click();", series[s_idx])
            time.sleep(12)

            for img_idx in range(1, 100):
                try:
                    # STEP 1: Wake up the Cornerstone Stack (Crucial Step)
                    # Trying to avoid "Consumers must define stacks" error
                    driver.execute_script("""
                        var canvas = document.querySelector('canvas, .viewport-element');
                        if (canvas) {
                            var rect = canvas.getBoundingClientRect();
                            var x = rect.left + rect.width / 2;
                            var y = rect.top + rect.height / 2;
                            canvas.dispatchEvent(new MouseEvent('mousedown', {bubbles: true, clientX: x, clientY: y}));
                            canvas.click();
                        }
                    """)
                    time.sleep(2)

                    # STEP 2: Open "More" Menu using the SVG path detected in DOM
                    driver.execute_script("""
                        var targetD = "M286.935 69.377c-3.614-3.617-7.898-5.424-12.848-5.424";
                        var path = Array.from(document.querySelectorAll('path')).find(p => p.getAttribute('d').includes(targetD));
                        if (path) {
                            var btn = path.closest('.toolbar-button') || path.parentElement;
                            var r = btn.getBoundingClientRect();
                            btn.dispatchEvent(new MouseEvent('click', {view: window, bubbles: true, clientX: r.right - 2, clientY: r.bottom - 2}));
                        }
                    """)
                    time.sleep(3)

                    # STEP 3: Click Download Button (data-cy="download")
                    btn_dl = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, '[data-cy="download"]')))
                    driver.execute_script("arguments[0].click();", btn_dl)
                    time.sleep(3)

                    # STEP 4: Set Resolution to 2000px
                    driver.execute_script("""
                        var inputs = document.querySelectorAll('input[type="number"]');
                        inputs.forEach(i => {
                            i.value = '2000';
                            i.dispatchEvent(new Event('input', { bubbles: true }));
                            i.dispatchEvent(new Event('change', { bubbles: true }));
                        });
                    """)
                    time.sleep(2)

                    # STEP 5: Confirm Download
                    btn_confirm = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Descargar')]")))
                    driver.execute_script("arguments[0].click();", btn_confirm)

                    print(f"   🖼️ Image {img_idx}: OK")
                    time.sleep(15) # Processing time for high-res blob

                    # Cleanup and next image
                    webdriver.ActionChains(driver).send_keys(Keys.ESCAPE).perform()
                    time.sleep(1)
                    webdriver.ActionChains(driver).send_keys(Keys.ARROW_DOWN).perform()
                    time.sleep(2)

                except Exception as e:
                    print(f"   🖼️ Image {img_idx}: Failed")
                    break
    finally:
        driver.quit()

if __name__ == "__main__":
    download_medical_images_anonymized(TARGET_URL)